# Listener Prior (Dual Dataset) - Colab GPU Training (Public Repo)

This notebook:
- clones the repo into `/content/listener-prior`
- installs dependencies (pins `datasets<4.0.0` so MultiWOZ/DailyDialog script datasets load)
- loads Hugging Face token from Colab Secrets (`HF_TOKEN`)
- trains on **MultiWOZ 2.2 + DailyDialog**
- builds examples to predict what the **other speaker** will say next (target role)
- evaluates on the **test** split each epoch and saves `encoder_best/` when improved
- writes outputs to Google Drive so they persist


In [ ]:
# --- CONFIG: set your GitHub repo here ---
REPO_URL = "https://github.com/<YOUR_GITHUB_USERNAME>/<YOUR_REPO_NAME>.git"  # <- edit
PROJECT_DIR = "/content/listener-prior"

In [ ]:
!rm -rf "$PROJECT_DIR"
!git clone "$REPO_URL" "$PROJECT_DIR"
%cd "$PROJECT_DIR"
!ls -la

In [ ]:
# Debug instrumentation: capture pre-install package versions
import importlib.metadata as _md
import json
import os
import sys
import time

_LOG_PATH = "/Users/erhanbilal/Documents/New project/.cursor/debug.log"


def _ver(pkg: str):
    try:
        return _md.version(pkg)
    except Exception:
        return None


def _log(hypothesis_id: str, message: str, data: dict):
    payload = {
        "sessionId": "debug-session",
        "runId": "preinstall",
        "hypothesisId": hypothesis_id,
        "location": "colab_train_gpu_public_dual_dataset.ipynb:preinstall",
        "message": message,
        "data": data,
        "timestamp": int(time.time() * 1000),
    }
    try:
        with open(_LOG_PATH, "a", encoding="utf-8") as f:
            f.write(json.dumps(payload) + "\n")
    except Exception as exc:
        print("debug log write failed:", exc)

# region agent log
_log(
    "H1",
    "preinstall_versions",
    {
        "python": sys.version.split()[0],
        "datasets": _ver("datasets"),
        "transformers": _ver("transformers"),
        "sentence_transformers": _ver("sentence-transformers"),
        "numpy": _ver("numpy"),
        "pandas": _ver("pandas"),
        "torch": _ver("torch"),
        "requests": _ver("requests"),
    },
)
# endregion

# region agent log
req_text = ""
try:
    req_text = open("requirements.txt", "r", encoding="utf-8").read()
except Exception as exc:
    req_text = f"ERROR_READ_REQUIREMENTS: {exc}"
_log(
    "H2",
    "requirements_snapshot",
    {
        "requirements_len": len(req_text.splitlines()),
        "requirements_preview": "\n".join(req_text.splitlines()[:10]),
    },
)
# endregion

In [ ]:
# Install deps. We intentionally do NOT pin torch here; Colab already has a CUDA build.
import pathlib
import json
import time

_LOG_PATH = "/Users/erhanbilal/Documents/New project/.cursor/debug.log"

def _log(hypothesis_id: str, message: str, data: dict):
    payload = {
        "sessionId": "debug-session",
        "runId": "install",
        "hypothesisId": hypothesis_id,
        "location": "colab_train_gpu_public_dual_dataset.ipynb:install",
        "message": message,
        "data": data,
        "timestamp": int(time.time() * 1000),
    }
    try:
        with open(_LOG_PATH, "a", encoding="utf-8") as f:
            f.write(json.dumps(payload) + "\n")
    except Exception as exc:
        print("debug log write failed:", exc)

req = pathlib.Path("requirements.txt").read_text().splitlines()
req_no_torch = [r for r in req if r.strip() and not r.strip().startswith("torch")]
pathlib.Path("/tmp/requirements_no_torch.txt").write_text("\n".join(req_no_torch) + "\n")

# region agent log
_log(
    "H2",
    "requirements_no_torch_preview",
    {
        "requirements_no_torch_len": len(req_no_torch),
        "requirements_no_torch_preview": "\n".join(req_no_torch[:15]),
    },
)
# endregion

!python -m pip install -U pip
!python -m pip install -r /tmp/requirements_no_torch.txt --upgrade --force-reinstall

import datasets
print("datasets:", datasets.__version__)
# region agent log
_log(
    "H3",
    "datasets_version_postinstall",
    {"datasets_version": datasets.__version__},
)
# endregion
assert tuple(int(x) for x in datasets.__version__.split(".")[:1]) < (4,), "datasets must be < 4.0.0; restart runtime after install"

In [ ]:
# If the assert above failed, go to Runtime -> Restart runtime, then rerun from the top.
# region agent log
try:
    import importlib.metadata as _md
    import json
    import time
    _LOG_PATH = "/Users/erhanbilal/Documents/New project/.cursor/debug.log"
    def _ver(pkg: str):
        try:
            return _md.version(pkg)
        except Exception:
            return None
    payload = {
        "sessionId": "debug-session",
        "runId": "postinstall",
        "hypothesisId": "H1",
        "location": "colab_train_gpu_public_dual_dataset.ipynb:postinstall",
        "message": "postinstall_versions",
        "data": {
            "datasets": _ver("datasets"),
            "transformers": _ver("transformers"),
            "sentence_transformers": _ver("sentence-transformers"),
            "numpy": _ver("numpy"),
            "pandas": _ver("pandas"),
            "torch": _ver("torch"),
            "requests": _ver("requests"),
            "packaging": _ver("packaging"),
        },
        "timestamp": int(time.time() * 1000),
    }
    with open(_LOG_PATH, "a", encoding="utf-8") as f:
        f.write(json.dumps(payload) + "\n")
except Exception as exc:
    print("debug log write failed:", exc)
# endregion
pass

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Hugging Face token (recommended).
# In Colab: Tools -> Secrets -> add HF_TOKEN.
import os
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
except Exception:
    token = None

# Clear any stale/expired tokens that might already exist in the environment.
for k in ['HF_TOKEN', 'HUGGINGFACE_HUB_TOKEN', 'HUGGING_FACE_HUB_TOKEN']:
    os.environ.pop(k, None)

if token:
    os.environ['HF_TOKEN'] = token
    os.environ['HUGGINGFACE_HUB_TOKEN'] = token
    os.environ['HUGGING_FACE_HUB_TOKEN'] = token
    print('HF token set from Colab Secrets.')
else:
    print('No HF_TOKEN secret found. Public datasets/models should still work without it.')

In [ ]:
import datetime
run_id = datetime.datetime.now().strftime('dual_run_%Y%m%d_%H%M%S')
OUTPUT_DIR = f"/content/drive/MyDrive/listener_prior_runs/{run_id}"
print('OUTPUT_DIR:', OUTPUT_DIR)

In [ ]:
# Train. This evaluates on TEST each epoch and updates encoder_best/ when improved.
# target_role=SYSTEM means: predict what the other person will say next.
!python scripts/train_dual_epoch_test.py \
  --output_dir "$OUTPUT_DIR" \
  --device auto \
  --epochs 6 \
  --batch_size 32 \
  --learning_rate 4.3e-5 \
  --weight_decay 0.01 \
  --adam_beta1 0.95 \
  --adam_beta2 0.98 \
  --adam_eps 1e-8 \
  --grad_accum_steps 2 \
  --warmup_ratio 0.0 \
  --history_turns 6 \
  --target_role SYSTEM \
  --val_ratio 0.05 \
  --test_ratio 0.15 \
  --max_dialogs_multiwoz 0 \
  --max_dialogs_dailydialog 0

In [ ]:
# Quick offline demo using the best encoder.
RUN_DIR = OUTPUT_DIR
!python -m src.demo_offline --run "$RUN_DIR" --encoder_subdir encoder_best